In [ ]:
# bulk_import hogdb to import comment and tag nodes + hasTag edges

import csv
import pandas as pd
from HOGDB.db.neo4j import Neo4jDatabase
from HOGDB.graph.graph_with_subgraph_storage import GraphwithSubgraphStorage
from HOGDB.db.schema import Schema
from HOGDB.db.label import Label

# ----- those files must be in the import folder of neo4j BD --
COMMENT_CSV = "comment_0_0.csv"                # doit etre dans le dossier import de la DB
TAG_CSV     = "tag_0_0.csv"                    
HAS_TAG_CSV = "comment_hasTag_tag_0_0.csv"     

DELIM = "|"   # vos CSVs utilisent '|' d'après votre code

# ---- schémas (mapping champs CSV -> propriétés de noeud) -----------
comment_node_schema = [
    Schema("id", int, "id"),
    Schema("creationDate", int, "creationDate"),
    Schema("locationIP", str, "locationIP"),
    Schema("browserUsed", str, "browserUsed"),
    Schema("length", int, "length"),
]

tag_node_schema = [
    Schema("id", int, "id"),
    Schema("name", str, "name"),
    Schema("url", str, "url"),
]

# schémas utilisés pour matcher lors de l'import des node-edges (nom de champs exactement comme dans le CSV HAS_TAG)
start_schema_for_has_tag = [ Schema("id", int, "Comment.id") ]
end_schema_for_has_tag   = [ Schema("id", int, "Tag.id") ]

# --- connexion & objet HO-GDB ---
db = Neo4jDatabase()                      
gs = GraphwithSubgraphStorage(db)         


    
    # importer les nœuds Comment (en batch côté serveur)
print("import des Comment nodes depuis CSV (bulk LOAD CSV)")
gs.import_nodes_from_csv(
        COMMENT_CSV,
        labels=[Label("Comment")],
        node_schema=comment_node_schema,
        delimiter=DELIM,
        as_url=False
    )

# importer les nœuds Tag
print("import des Tag nodes depuis CSV (bulk LOAD CSV)")
gs.import_nodes_from_csv(
        TAG_CSV,
        labels=[Label("Tag")],
        node_schema=tag_node_schema,
        delimiter=DELIM,
        as_url=False
    )

# importer les HAS_TAG edges  comme node-edge (créer un nœud _edge + 2 _adjacency)
   
print(" import des HAS_TAG ")
gs.import_edges_from_csv(
    HAS_TAG_CSV,
    start_node_labels =[Label("Comment")],
    start_node_schema=start_schema_for_has_tag,
    end_node_labels=[Label("Tag")],
    end_node_schema=end_schema_for_has_tag,
    edge_label=Label("HAS_TAG"),
    edge_schema=[],   # pas de propriétés
    as_url=False,
    delimiter="|"
)

   

In [ ]:
 # construire le CSV des subgraphs "taggedComment" :
    #    chaque ligne représente une sous-collection contenant les deux noeuds (comment;tag) et l'arête (comment:tag)
    #    Format attendu par import_subgraphs_from_csv:
    #        header: <node_field_name>|<edge_field_name>[|...other subgraph props...]
    #    Ici j'utilise "nodes" et "edges" comme noms de champs utilisés dans Schema() plus bas.
print(" génération du CSV de subgraphs (taggedComment)")
with open(HAS_TAG_CSV, newline='', encoding="utf-8") as fin, open(SUBGRAPH_CSV, "w", newline='', encoding="utf-8") as fout:
        rdr = csv.DictReader(fin, delimiter=DELIM)
        writer = csv.writer(fout, delimiter=DELIM, quoting=csv.QUOTE_MINIMAL)
        # en-tête : nodes | edges  (vous pouvez ajouter d'autres colonnes pour les propriétés du subgraph)
        writer.writerow(["nodes", "edges"])
        for row in rdr:
            c_id = row.get("Comment.id") or row.get("commentId") or row.get("CommentId")
            t_id = row.get("Tag.id") or row.get("tagId") or row.get("TagId")
            if not c_id or not t_id:
                continue
            nodes_field = f"{c_id};{t_id}"      # liste des noeuds (séparateur ';')
            edges_field = f"{c_id}:{t_id}"      # chaque arête référencée comme start:end
            writer.writerow([nodes_field, edges_field])

   

In [ ]:
 #  import les subgraphs via import_subgraphs_from_csv 
# immmportant : once the taggedComment_subgraphs.csv is created , it must be moved to the import folder of your BD in neo4j

print("import des subgraph collections taggedComment")


SUBGRAPH_CSV = "taggedComment_subgraphs.csv"   # généré par le script


node_schema_for_subgraph = Schema("id", int, "nodes")   # field 'nodes' dans le CSV contient les id séparés par ';'
edge_schema_for_subgraph = Schema("id", int, "edges")   # field 'edges' contient start:end entries

db.import_subgraphs_from_csv(
        gs.session,
        SUBGRAPH_CSV,
        node_schema=node_schema_for_subgraph,
        edge_schema=edge_schema_for_subgraph,
        common_schema=[],                        # si vous avez propriétés communes aux nodes, les déclarer ici
        subgraph_labels=[Label("taggedComment")],
        subgraph_schema=[],                      # propriétés du subgraph si vous avez (sinon [])
        as_url=False,
        delimiter=DELIM,
        batch_size=1000
    )




 
print("fermeture connexion DB")
gs.close_connection()
